# 02 — Data Preparation

Reads raw parquet files produced by CarpDataStream, applies cleaning, and writes a prepared dataset to `data/processed/`.

**Steps:**
1. Filter all sensor streams to the conference week (Oct 13–16, inclusive).
2. Convert cumulative step counts to per-interval deltas.
3. Parse surveys into structured tables (onboarding, daily stress/sleep, WHO-5, BFI-10).
4. Write everything to `data/processed/` as parquet files.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# ── Paths ──
PROJECT_ROOT = Path.cwd().parent
RAW_PARQUET = PROJECT_ROOT / "data" / "raw" / "parquet"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data.surveys import (
    load_submitted,
    parse_daily_sleep,
    parse_daily_stress,
    parse_onboarding,
    parse_who5,
    parse_bfi10,
    parse_event_ratings,
    parse_reaction_time,
    parse_flanker,
)
from data.time import add_time_columns

print(f"Raw parquets: {RAW_PARQUET}")
print(f"Output dir:   {OUT_DIR}")
print(f"Available:    {sorted(f.stem for f in RAW_PARQUET.glob('*.parquet'))}")

Raw parquets: /Users/mikloshavlik/Study/UbiLife/data/raw/parquet
Output dir:   /Users/mikloshavlik/Study/UbiLife/data/processed
Available:    ['activity', 'airquality', 'ambientlight', 'batterystate', 'completedapptask', 'deviceinformation', 'error', 'heartbeat', 'image', 'location', 'screenevent', 'stepcount', 'survey', 'triggeredtask', 'weather']


## 1. Filter Sensor Streams to Conference Week (Oct 13–16)

Discard all sensor data outside the main conference dates to focus analysis on the event itself.

In [2]:
CONF_START = pd.Timestamp("2025-10-13")
CONF_END = pd.Timestamp("2025-10-16 23:59:59.999999")

SENSOR_STREAMS = [
    "activity", "airquality", "ambientlight", "batterystate",
    "heartbeat", "location", "screenevent", "stepcount", "weather",
]


def load_raw(name: str) -> pd.DataFrame:
    """Load a raw parquet with local (Europe/Helsinki) timestamp + participant_id."""
    return add_time_columns(pd.read_parquet(RAW_PARQUET / f"{name}.parquet"))


def filter_conference_week(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows within Oct 13-16 2025 local time (inclusive)."""
    mask = (df["timestamp"] >= CONF_START) & (df["timestamp"] <= CONF_END)
    return df.loc[mask].copy()


results = []
for name in SENSOR_STREAMS:
    src = RAW_PARQUET / f"{name}.parquet"
    if not src.exists():
        print(f"SKIP {name} (not found)")
        continue

    df = load_raw(name)
    n_before = len(df)
    df = filter_conference_week(df)
    n_after = len(df)

    # Save filtered (stepcount handled separately in next cell)
    if name != "stepcount":
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)
    else:
        df_steps = df  # keep for next cell

    results.append({
        "stream": name,
        "before": n_before,
        "after": n_after,
        "kept_pct": n_after / n_before * 100,
        "participants": df["participant_id"].nunique(),
    })

filter_df = pd.DataFrame(results)
filter_df["kept_pct"] = filter_df["kept_pct"].round(1)
filter_df

,stream,before,after,kept_pct,participants
0,activity,199336,166047,83.3,77
1,airquality,3837,3033,79.0,65
2,ambientlight,3115,1993,64.0,19
3,batterystate,18119,13263,73.2,58
4,heartbeat,204811,106244,51.9,80
5,location,133082,93209,70.0,78
6,screenevent,5657,5606,99.1,22
7,stepcount,447625,393300,87.9,77
8,weather,3854,3050,79.1,65


## 2. Convert Cumulative Step Counts to Deltas

The raw step count is a monotonically increasing pedometer counter. We compute per-interval deltas (diff), clamping negative values (pedometer resets) to zero.

In [3]:
df_steps = df_steps.sort_values(["participant_id", "timestamp"])

df_steps["steps_cumulative"] = df_steps["measurement.data.steps"]
df_steps["steps_delta"] = (
    df_steps.groupby("participant_id")["steps_cumulative"]
    .diff()
    .clip(lower=0)
)
df_steps["steps_delta"] = df_steps["steps_delta"].fillna(0).astype(int)

df_steps.to_parquet(OUT_DIR / "stepcount.parquet", index=False)

# Show a sample
pid = df_steps["participant_id"].value_counts().index[0]
sample = df_steps[df_steps["participant_id"] == pid].head(10)
print(f"Sample deltas for participant {pid[:12]}...")
sample[["timestamp", "steps_cumulative", "steps_delta"]]

Sample deltas for participant b403e2f9-0be...


,timestamp,steps_cumulative,steps_delta
69912,2025-10-13 08:07:23.657356,15375,0
69913,2025-10-13 08:07:24.057580,15376,1
69914,2025-10-13 08:07:24.544699,15377,1
69915,2025-10-13 08:09:04.068720,15384,7
69916,2025-10-13 08:09:04.390789,15385,1
69917,2025-10-13 08:09:05.034738,15386,1
69918,2025-10-13 08:09:17.826594,15407,21
69919,2025-10-13 08:09:22.004190,15408,1
69920,2025-10-13 08:09:22.008288,15413,5
69921,2025-10-13 08:09:57.967036,15428,15


## 3. Parse Surveys into Structured Tables

Each survey type gets its own clean DataFrame:

| Table | Shape | Key columns |
|---|---|---|
| `survey_onboarding` | 1 row/participant | age, gender, academic role, presenter types |
| `survey_daily_stress` | 1 row/participant/day | stress_level (0–10) |
| `survey_daily_sleep` | 1 row/participant/day | sleep quality + hours |
| `survey_who5` | 1 row/participant | q1–q5, raw_sum, pct_score |
| `survey_bfi10` | 1 row/participant | Big Five dimensions |
| `survey_event_ratings` | 1 row/participant/event | session (A–H) and keynote (1–3) ratings |

In [4]:
sub = load_submitted(RAW_PARQUET)

survey_parsers = {
    "survey_daily_stress": parse_daily_stress,
    "survey_daily_sleep": parse_daily_sleep,
    "survey_onboarding": parse_onboarding,
    "survey_who5": parse_who5,
    "survey_bfi10": parse_bfi10,
    "survey_event_ratings": parse_event_ratings,
    "cognition_reaction_time": parse_reaction_time,
    "cognition_flanker": parse_flanker,
}

survey_tables = {}
summary = []

for table_name, parser in survey_parsers.items():
    df = parser(sub)
    df.to_parquet(OUT_DIR / f"{table_name}.parquet", index=False)
    survey_tables[table_name] = df
    summary.append({
        "table": table_name,
        "rows": len(df),
        "participants": df["participant_id"].nunique(),
        "columns": ", ".join(c for c in df.columns if c not in ["participant_id", "timestamp", "date"]),
    })

pd.DataFrame(summary)

,table,rows,participants,columns
0,survey_daily_stress,70,36,stress_level
1,survey_daily_sleep,64,36,"sleep_quality_value, sleep_quality_text, sleep..."
2,survey_onboarding,38,38,"age, gender_value, gender_text, academic_value..."
3,survey_who5,37,37,"q1, q2, q3, q4, q5, raw_sum, pct_score"
4,survey_bfi10,42,42,"extraversion, agreeableness, conscientiousness..."
5,survey_event_ratings,205,38,"attending_text, rating_value, rating_text, eve..."
6,cognition_reaction_time,52,26,"avg_reaction_time_ms, correct_taps, wrong_taps"
7,cognition_flanker,35,23,"score, time, right_swipes, wrong_swipes, mean_..."


In [5]:
for name, df in survey_tables.items():
    print(f"── {name} ──")
    display(df.head())
    print()

── survey_daily_stress ──


,participant_id,date,timestamp,stress_level
7,051da686-748f-4343-bd85-870f579bd485,2025-10-16,2025-10-16 18:20:20.140230,7.0
26,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14,2025-10-14 22:40:19.061501,5.0
46,52ba4418-137f-4e45-9089-ca5f71783ce8,2025-10-14,2025-10-14 16:00:20.709400,4.0
47,aaa33240-0619-4dff-91fd-e36e2ad08c14,2025-10-14,2025-10-14 16:00:10.522427,1.0
48,ccb3d959-320c-48bd-80f2-e9624f41f376,2025-10-14,2025-10-14 16:05:59.211226,7.0



── survey_daily_sleep ──


,participant_id,date,timestamp,sleep_quality_value,sleep_quality_text,sleep_hours_value,sleep_hours_text
76,72405115-4aec-4a56-b78e-ca21fd75dc18,2025-10-16,2025-10-16 10:38:00.954374,2,Neutral,3,<6 hours
91,f7545e44-04d8-4daf-9e16-805b5913b801,2025-10-14,2025-10-14 18:13:50.809450,3,Good,2,6-8 hours
93,051da686-748f-4343-bd85-870f579bd485,2025-10-14,2025-10-14 18:22:11.102862,0,Very Bad,3,<6 hours
111,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-16,2025-10-16 10:43:42.690059,3,Good,2,6-8 hours
113,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-16,2025-10-16 10:43:54.617317,1,Bad,3,<6 hours



── survey_onboarding ──


...

,participant_id,timestamp,attending_text,rating_value,rating_text,event_type,event_id
0,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-14 14:06:10.916310,A3: Mental Health,3,Interesting,session,A
1,e7cb43d0-f5ac-446d-932e-31c349076dd5,2025-10-14 14:31:24.280974,A3: Mental Health,3,Interesting,session,A
2,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14 22:41:42.686959,A3: Mental Health,3,Interesting,session,A
3,5303dfda-cc84-41f3-a776-9a2d8f805651,2025-10-14 15:35:30.350878,"A1: Smart Homes & Textiles, A3: Mental Health",4,Amazing,session,A
4,4954fe95-1abd-4fb0-a021-8676c5527f3c,2025-10-14 16:07:03.672313,"A1: Smart Homes & Textiles, A4: Augmented & Vi...",2,Neutral,session,A



── cognition_reaction_time ──


,participant_id,date,timestamp,avg_reaction_time_ms,correct_taps,wrong_taps
166,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-13,2025-10-13 14:42:24.603074,920.0,7.0,0.0
167,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-13,2025-10-13 14:43:05.335426,434.0,8.0,0.0
221,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-16,2025-10-16 10:43:59.545062,416.0,9.0,0.0
223,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-16,2025-10-16 10:44:47.178511,440.0,7.0,1.0
636,051da686-748f-4343-bd85-870f579bd485,2025-10-16,2025-10-16 15:22:01.709223,385.0,9.0,0.0



── cognition_flanker ──


,participant_id,date,timestamp,score,time,right_swipes,wrong_swipes,mean_congruent_ms,mean_incongruent_ms,n_congruent,n_incongruent
415,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-15,2025-10-15 11:27:25.389105,1.0,8.0,10.0,0.0,933.833333,813.75,6.0,4.0
222,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-16,2025-10-16 10:44:10.395139,1.0,7.0,10.0,0.0,751.500000,823.00,6.0,4.0
3,051da686-748f-4343-bd85-870f579bd485,2025-10-16,2025-10-16 17:35:29.794573,1.0,8.0,10.0,0.0,736.000000,1219.00,8.0,2.0
6,051da686-748f-4343-bd85-870f579bd485,2025-10-16,2025-10-16 17:36:02.363739,1.0,7.0,10.0,0.0,661.400000,796.00,5.0,5.0
446,1102dc18-32a5-4df6-84e2-403e60ab9e59,2025-10-15,2025-10-15 13:03:40.524535,1.0,7.0,10.0,0.0,739.500000,744.00,4.0,6.0


## 4. Summary

In [6]:
all_files = sorted(OUT_DIR.glob("*.parquet"))

file_summary = []
for f in all_files:
    df = pd.read_parquet(f)
    file_summary.append({
        "file": f.name,
        "rows": len(df),
        "columns": len(df.columns),
        "size_kb": f.stat().st_size // 1024,
    })

summary_df = pd.DataFrame(file_summary)
print(f"Output directory: {OUT_DIR}")
print(f"Total files: {len(all_files)}")
print(f"Total size: {summary_df['size_kb'].sum():,} KB")
print()
summary_df

Output directory: /Users/mikloshavlik/Study/UbiLife/data/processed
Total files: 17
Total size: 22,019 KB



,file,rows,columns,size_kb
0,activity.parquet,166047,17,3055
1,airquality.parquet,3033,21,99
2,ambientlight.parquet,1993,20,95
3,batterystate.parquet,13263,17,307
4,cognition_flanker.parquet,35,11,8
5,cognition_reaction_time.parquet,52,6,6
6,heartbeat.parquet,106244,18,1772
7,location.parquet,93209,28,7727
8,screenevent.parquet,5606,16,133
9,stepcount.parquet,393300,18,8608
